## 🥈 Silver Layer - Data Cleaning & Feature Engineering
This notebook reads from the Bronze Delta table, cleans the data, fixes data types, extracts time-based and geo features, and saves a clean enriched Delta table ready for ML and analytics.

In [0]:
# Read from Bronze Delta table
df_bronze = spark.read.format("delta").table("workspace.default.bronze_crimes")

print(f"Records loaded from Bronze: {df_bronze.count()}")
display(df_bronze.limit(5))

Records loaded from Bronze: 1286734


ID,Case_Number,Date,Block,IUCR,Primary_Type,Description,Location_Description,Arrest,Domestic,Beat,District,Ward,Community_Area,FBI_Code,X_Coordinate,Y_Coordinate,Year,Updated_On,Latitude,Longitude,Location
14193417,JK249977,05/11/2026 12:00:00 AM,058XX N ROGERS AVE,0710,THEFT,THEFT FROM MOTOR VEHICLE,PARKING LOT / GARAGE (NON RESIDENTIAL),false,false,1711,17,39,13,06,1146681,1938590,2026,05/18/2026 03:42:59 PM,41.987468468,-87.735867509,"(41.987468468, -87.735867509)"
14193533,JK250272,05/11/2026 12:00:00 AM,085XX S HERMITAGE AVE,0760,BURGLARY,BURGLARY FROM MOTOR VEHICLE,STREET,false,false,614,6,18,71,06,1166163,1848161,2026,05/18/2026 03:42:59 PM,41.738928101,-87.666794079,"(41.738928101, -87.666794079)"
14193357,JK249936,05/11/2026 12:00:00 AM,057XX W CHICAGO AVE,0610,BURGLARY,FORCIBLE ENTRY,COMMERCIAL / BUSINESS OFFICE,false,false,1511,15,29,25,05,1137755,1904757,2026,05/18/2026 03:42:59 PM,41.894793399,-87.769516687,"(41.894793399, -87.769516687)"
14197599,JK254317,05/11/2026 12:00:00 AM,028XX N BURLING ST,0820,THEFT,$500 AND UNDER,APARTMENT,false,false,1934,19,44,6,06,1170810,1919094,2026,05/18/2026 03:42:59 PM,41.933474653,-87.647693928,"(41.933474653, -87.647693928)"
14194155,JK250986,05/11/2026 12:00:00 AM,006XX E 112TH ST,0890,THEFT,FROM BUILDING,RESIDENCE - YARD (FRONT / BACK),false,false,531,5,9,50,06,1182739,1830829,2026,05/18/2026 03:42:59 PM,41.690998601,-87.6065992,"(41.690998601, -87.6065992)"


### 🧹 Fix Data Types & Drop Useless Columns
Convert Date from string to timestamp, drop the redundant Location column, and cast correct data types.

In [0]:
from pyspark.sql.functions import to_timestamp, col

# Convert Date from string to timestamp
# Drop redundant Location column
df_typed = df_bronze \
    .withColumn("Date", to_timestamp(col("Date"), "MM/dd/yyyy hh:mm:ss a")) \
    .withColumn("Updated_On", to_timestamp(col("Updated_On"), "MM/dd/yyyy hh:mm:ss a")) \
    .drop("Location")

print(f"Columns after cleaning: {len(df_typed.columns)}")
df_typed.printSchema()

Columns after cleaning: 21
root
 |-- ID: integer (nullable = true)
 |-- Case_Number: string (nullable = true)
 |-- Date: timestamp (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary_Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location_Description: string (nullable = true)
 |-- Arrest: boolean (nullable = true)
 |-- Domestic: boolean (nullable = true)
 |-- Beat: integer (nullable = true)
 |-- District: integer (nullable = true)
 |-- Ward: integer (nullable = true)
 |-- Community_Area: integer (nullable = true)
 |-- FBI_Code: string (nullable = true)
 |-- X_Coordinate: integer (nullable = true)
 |-- Y_Coordinate: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Updated_On: timestamp (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)



### ⏰ Extract Time-Based Features
Extract hour, day of week, month and season from the Date column. These will be key features for our ML models.

In [0]:
from pyspark.sql.functions import hour, dayofweek, month, when

df_time = df_typed \
    .withColumn("Hour", hour(col("Date"))) \
    .withColumn("Day_Of_Week", dayofweek(col("Date"))) \
    .withColumn("Month", month(col("Date"))) \
    .withColumn("Season", 
        when((month(col("Date")).isin(12, 1, 2)), "Winter")
        .when((month(col("Date")).isin(3, 4, 5)), "Spring")
        .when((month(col("Date")).isin(6, 7, 8)), "Summer")
        .otherwise("Fall")) \
    .withColumn("Is_Weekend",
        when(dayofweek(col("Date")).isin(1, 7), True)
        .otherwise(False)) \
    .withColumn("Is_Night",
        when((hour(col("Date")) >= 20) | (hour(col("Date")) <= 6), True)
        .otherwise(False))

print(f"New time features added! Total columns: {len(df_time.columns)}")
display(df_time.select("Date", "Hour", "Day_Of_Week", "Month", "Season", "Is_Weekend", "Is_Night").limit(10))

New time features added! Total columns: 27


Date,Hour,Day_Of_Week,Month,Season,Is_Weekend,Is_Night
2023-06-25T20:20:00.000Z,20,1,6,Summer,true,true
2023-06-25T20:20:00.000Z,20,1,6,Summer,true,true
2023-06-25T20:11:00.000Z,20,1,6,Summer,true,true
2023-06-25T20:08:00.000Z,20,1,6,Summer,true,true
2023-06-25T20:06:00.000Z,20,1,6,Summer,true,true
2023-06-25T20:01:00.000Z,20,1,6,Summer,true,true
2023-06-25T20:00:00.000Z,20,1,6,Summer,true,true
2023-06-25T20:00:00.000Z,20,1,6,Summer,true,true
2023-06-25T20:00:00.000Z,20,1,6,Summer,true,true
2023-06-25T20:00:00.000Z,20,1,6,Summer,true,true


### 🧹 Handle Null Values
Check and handle missing values across key columns before saving to Silver.

In [0]:
from pyspark.sql.functions import count, isnan, when, col

# Check nulls across all columns
print("🔍 Null counts per column:")
df_time.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in df_time.columns
]).show(vertical=True)

🔍 Null counts per column:
-RECORD 0---------------------
 ID                   | 0     
 Case_Number          | 0     
 Date                 | 0     
 Block                | 0     
 IUCR                 | 0     
 Primary_Type         | 0     
 Description          | 0     
 Location_Description | 6551  
 Arrest               | 0     
 Domestic             | 0     
 Beat                 | 0     
 District             | 0     
 Ward                 | 25    
 Community_Area       | 114   
 FBI_Code             | 0     
 X_Coordinate         | 17244 
 Y_Coordinate         | 17244 
 Year                 | 0     
 Updated_On           | 0     
 Latitude             | 17244 
 Longitude            | 17244 
 Hour                 | 0     
 Day_Of_Week          | 0     
 Month                | 0     
 Season               | 0     
 Is_Weekend           | 0     
 Is_Night             | 0     



### 🔧 Fix Null Values
Drop rows with missing coordinates since location is critical for hotspot prediction. Fill remaining nulls with sensible defaults.

In [0]:
# Drop rows where location is null (critical for hotspot prediction)
# Fill remaining nulls with sensible defaults
df_clean = df_time \
    .dropna(subset=["Latitude", "Longitude", "X_Coordinate", "Y_Coordinate"]) \
    .fillna({
        "Location_Description": "Unknown",
        "Ward": 0,
        "Community_Area": 0
    })

print(f"Records before null handling: {df_time.count()}")
print(f"Records after null handling:  {df_clean.count()}")
print(f"Records dropped:              {df_time.count() - df_clean.count()}")

Records before null handling: 1286734
Records after null handling:  1269490
Records dropped:              17244


### 💾 Save to Silver Delta Table
Save the cleaned and feature-enriched dataset as a Silver Delta table, ready for ML models and Gold layer analytics.

In [0]:
# Save as Silver Delta Table
df_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_crimes")

print("✅ Silver Delta table created successfully!")
print(f"Total clean records: {df_clean.count()}")
print(f"Total features: {len(df_clean.columns)}")

✅ Silver Delta table created successfully!
Total clean records: 1269490
Total features: 27


### ✅ Verify Silver Table
Confirm the clean data landed correctly in our Silver Delta table.

In [0]:
df_verify = spark.sql("""
    SELECT 
        COUNT(*) as total_records,
        COUNT(DISTINCT Primary_Type) as crime_types,
        COUNT(DISTINCT District) as districts,
        MIN(Date) as earliest_crime,
        MAX(Date) as latest_crime
    FROM workspace.default.silver_crimes
""")
display(df_verify)

total_records,crime_types,districts,earliest_crime,latest_crime
1269490,31,23,2021-01-01T00:00:00.000Z,2026-05-11T00:00:00.000Z
